# SpecSafe V5 Governed Kaggle Trace Collection

## Purpose

This notebook collects a **small, fixed, self-authored trace corpus** from the approved Qwen model pair after the retained T4 preflight pass.

It exports separate runtime-candidate records and post-hoc target-derived outcome records. Raw prompts, decoded tokens, full logits, private data, and credentials are never exported.

## Strict boundary

- Use **GPU T4 x2** and Internet.
- Replace only `SOURCE_COMMIT_SHA` after the notebook PR merges.
- Run all cells exactly once for attempt 001.
- Stop after downloading the result JSON and trace archive.
- Do not fit calibration, score policies, alter prompts, add prompts, or publish output from this notebook.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import shutil
from pathlib import Path
from typing import Any
from zipfile import ZIP_DEFLATED, ZipFile

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# Required after this notebook PR merges. Do not run with this value unchanged.
SOURCE_COMMIT_SHA = "RECORD_MAIN_COMMIT_SHA_AFTER_PR_MERGE"

COLLECTION_ID = "v5-qwen-governed-trace-collection-v1"
COLLECTION_ATTEMPT_ID = "attempt-001-t4"
RESULT_SCHEMA_VERSION = "specsafe-kaggle-trace-collection-result-v1"
MANIFEST_SCHEMA_VERSION = "specsafe-kaggle-trace-collection-manifest-v1"
RUNTIME_RECORD_SCHEMA_VERSION = "specsafe-kaggle-trace-runtime-record-v1"
OUTCOME_RECORD_SCHEMA_VERSION = "specsafe-kaggle-trace-expected-outcome-record-v1"

PREFLIGHT_ATTEMPT_ID = "attempt-003-t4-pass"
PREFLIGHT_RESULT_SHA256 = "4b2e096ecf57e7c729918a83a6b84434ab0dc9dac094f6b9eadff40f84e7d9dd"
PREFLIGHT_SOURCE_COMMIT_SHA = "061f58cba075cafbfbc4ed5d0afe9e817997a877"

DRAFT_MODEL_ID = "Qwen/Qwen2.5-0.5B"
DRAFT_MODEL_REVISION = "060db6499f32faf8b98477b0a26969ef7d8b9987"
TARGET_MODEL_ID = "Qwen/Qwen2.5-1.5B"
TARGET_MODEL_REVISION = "8faed761d45a263340a0528343f099c05c9a4323"
TOKENIZER_SOURCE_MODEL_ID = DRAFT_MODEL_ID
TOKENIZER_SOURCE_MODEL_REVISION = DRAFT_MODEL_REVISION
MODEL_PAIR_ID = "qwen2.5-base-0.5b-to-1.5b-v1"

SEED = 20260708
MAX_BLOCK_POSITIONS = 4
DECODING_CONFIGURATION_ID = "greedy-next-token-block-4-v1"
PROMPT_CORPUS_ID = "v5-qwen-self-authored-trace-corpus-v1"
PROMPT_CORPUS_SHA256 = "ffe698c9d9c41ea4a374ca7d12293130c832a7523f7554079314875afcce3d52"

WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIRECTORY = WORKING_ROOT / "specsafe_v5_qwen_trace_collection_v1_attempt_001"
STAGING_DIRECTORY = WORKING_ROOT / "specsafe_v5_qwen_trace_collection_v1_attempt_001_staging"
ARCHIVE_PATH = WORKING_ROOT / "specsafe_v5_qwen_trace_collection_v1_attempt_001.zip"
RESULT_PATH = WORKING_ROOT / "specsafe_v5_qwen_trace_collection_result_attempt_001.json"

# Self-authored public-safe prompts are hashed in exported artifacts.
# Raw text is never exported.
PROMPT_CASES = (
    {
        "case_id": "KTC5-001",
        "workload_type": "structured_text",
        "prompt": "Complete the checklist title: Safety review requires evidence, scope, and",
    },
    {
        "case_id": "KTC5-002",
        "workload_type": "structured_text",
        "prompt": "Write the next item in a short numbered plan for testing a small program:",
    },
    {
        "case_id": "KTC5-003",
        "workload_type": "code",
        "prompt": "def multiply(left, right):\n    return",
    },
    {
        "case_id": "KTC5-004",
        "workload_type": "code",
        "prompt": "for item in values:\n    if item > limit:\n        ",
    },
    {
        "case_id": "KTC5-005",
        "workload_type": "open_ended_chat",
        "prompt": "A traveler reached a quiet coastal town at dusk and noticed",
    },
    {
        "case_id": "KTC5-006",
        "workload_type": "open_ended_chat",
        "prompt": "Explain, in two short sentences, why a careful experiment records",
    },
)


class TraceCollectionFailure(RuntimeError):
    """Stops collection after retaining a bounded failure result."""

    def __init__(self, code: str, message: str) -> None:
        super().__init__(message)
        self.code = code


def canonical_json(value: Any) -> str:
    """Produce stable JSON for manifests and retained results."""

    return json.dumps(value, indent=2, sort_keys=True) + "\n"


def canonical_json_bytes(value: Any) -> bytes:
    """Return a compact canonical JSON encoding for hash identity."""

    return json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")


def sha256_bytes(value: bytes) -> str:
    """Return a SHA-256 digest for retained bytes."""

    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path) -> str:
    """Return a SHA-256 digest for one local artifact."""

    return sha256_bytes(path.read_bytes())


def safe_error_message(error: BaseException) -> str:
    """Keep retained failures bounded and avoid environment dumps."""

    return f"{type(error).__name__}: {str(error)[:400]}"


def require_source_commit_sha() -> str:
    """Require exact collection-notebook provenance."""

    if SOURCE_COMMIT_SHA == "RECORD_MAIN_COMMIT_SHA_AFTER_PR_MERGE":
        raise TraceCollectionFailure(
            "source_commit_sha_missing",
            "Record the merged main commit SHA in SOURCE_COMMIT_SHA before collection.",
        )
    if len(SOURCE_COMMIT_SHA) < 7:
        raise TraceCollectionFailure(
            "source_commit_sha_invalid",
            "SOURCE_COMMIT_SHA must contain at least seven commit characters.",
        )
    return SOURCE_COMMIT_SHA


def require_kaggle_t4() -> dict[str, Any]:
    """Require a supported CUDA architecture before model downloads or collection."""

    if not torch.cuda.is_available():
        raise TraceCollectionFailure(
            "gpu_unavailable",
            "Kaggle GPU is unavailable. Enable GPU T4 x2 before collection.",
        )
    capability_major, capability_minor = torch.cuda.get_device_capability(0)
    active_architecture = f"sm_{capability_major}{capability_minor}"
    supported_architectures = sorted(torch.cuda.get_arch_list())
    if active_architecture not in supported_architectures:
        raise TraceCollectionFailure(
            "gpu_architecture_unsupported",
            "Active GPU architecture is unsupported by the installed PyTorch build.",
        )
    if active_architecture != "sm_75":
        raise TraceCollectionFailure(
            "gpu_architecture_unsupported",
            "This governed attempt requires Kaggle GPU T4 x2 with architecture sm_75.",
        )
    return {
        "gpu_name": torch.cuda.get_device_name(0),
        "gpu_architecture": active_architecture,
        "torch_supported_gpu_architectures": supported_architectures,
    }


def choose_dtype() -> torch.dtype:
    """Choose the smallest safe dtype for the approved Kaggle T4 environment."""

    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def environment_metadata(gpu: dict[str, Any] | None = None) -> dict[str, Any]:
    """Retain bounded reproducibility metadata without paths or secrets."""

    metadata: dict[str, Any] = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown"),
        "kaggle_url_base": os.environ.get("KAGGLE_URL_BASE", "unknown"),
    }
    if gpu is not None:
        metadata.update(gpu)
    return metadata


def validate_prompt_corpus() -> None:
    """Reject notebook/corpus drift before model loading."""

    corpus = {
        "schema_version": "specsafe-kaggle-trace-prompt-corpus-v1",
        "corpus_id": PROMPT_CORPUS_ID,
        "data_role": "trace_collection",
        "source_type": "self_authored_public_safe",
        "case_count": len(PROMPT_CASES),
        "cases": PROMPT_CASES,
    }
    observed_sha256 = sha256_bytes(canonical_json_bytes(corpus))
    if observed_sha256 != PROMPT_CORPUS_SHA256:
        raise TraceCollectionFailure(
            "prompt_corpus_mismatch",
            "Notebook self-authored prompt corpus does not match its declared corpus hash.",
        )
    if len({case["case_id"] for case in PROMPT_CASES}) != len(PROMPT_CASES):
        raise TraceCollectionFailure(
            "prompt_corpus_mismatch",
            "Prompt corpus case IDs must be unique.",
        )


def require_clean_destination() -> None:
    """Preserve write-once attempt semantics for collection outputs."""

    for path in (OUTPUT_DIRECTORY, STAGING_DIRECTORY, ARCHIVE_PATH):
        if path.exists():
            raise TraceCollectionFailure(
                "output_already_exists",
                "Collection attempt destination already exists. Do not overwrite retained output.",
            )


def load_model(model_id: str, revision: str, dtype: torch.dtype) -> Any:
    """Load one pinned model at a time with no remote code execution."""

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        dtype=dtype,
        trust_remote_code=False,
    )
    model.to(torch.device("cuda"))
    model.eval()
    return model


def next_token_features(model: Any, input_ids: list[int]) -> dict[str, Any]:
    """Return finite next-token distribution features without retaining full logits."""

    tensor = torch.tensor([input_ids], dtype=torch.long, device=torch.device("cuda"))
    with torch.inference_mode():
        logits = model(input_ids=tensor).logits[:, -1, :]
    if logits.ndim != 2 or not bool(torch.isfinite(logits).all().item()):
        raise TraceCollectionFailure(
            "model_output_non_finite",
            "Model next-token logits are non-finite or have an unexpected shape.",
        )
    vocabulary_size = int(logits.shape[-1])
    model_vocabulary_size = int(model.config.vocab_size)
    if vocabulary_size != model_vocabulary_size:
        raise TraceCollectionFailure(
            "model_vocabulary_mismatch",
            "Observed model output vocabulary does not match model.config.vocab_size.",
        )
    log_probabilities = torch.log_softmax(logits.float(), dim=-1)
    probabilities = torch.exp(log_probabilities)
    token_id = int(torch.argmax(logits, dim=-1).item())
    probability = float(probabilities[0, token_id].item())
    entropy = float((-(probabilities * log_probabilities).sum(dim=-1)).item())
    return {
        "token_id": token_id,
        "probability": probability,
        "entropy": entropy,
        "vocabulary_size": vocabulary_size,
    }


def collect_case(
    *,
    case: dict[str, str],
    tokenizer: Any,
    draft_model: Any,
    target_model: Any,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Collect one bounded greedy block while preserving runtime/outcome separation."""

    prompt = case["prompt"]
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    if not prompt_ids:
        raise TraceCollectionFailure(
            "prompt_corpus_mismatch",
            "A self-authored prompt encoded to zero tokens.",
        )
    prompt_sha256 = sha256_bytes(prompt.encode("utf-8"))
    visible_prefix_token_ids: list[int] = []
    runtime_records: list[dict[str, Any]] = []
    outcome_records: list[dict[str, Any]] = []
    target_match_history: list[bool] = []

    for position in range(1, MAX_BLOCK_POSITIONS + 1):
        model_input_ids = prompt_ids + visible_prefix_token_ids
        draft_features = next_token_features(draft_model, model_input_ids)
        target_features = next_token_features(target_model, model_input_ids)
        candidate_token_id = draft_features["token_id"]
        target_argmax_matches_candidate = target_features["token_id"] == candidate_token_id
        target_match_history.append(target_argmax_matches_candidate)
        trace_id = f"{case['case_id']}-trace"
        request_id = f"{case['case_id']}-request"

        runtime_records.append(
            {
                "schema_version": RUNTIME_RECORD_SCHEMA_VERSION,
                "trace_id": trace_id,
                "case_id": case["case_id"],
                "request_id": request_id,
                "workload_type": case["workload_type"],
                "data_role": "trace_collection",
                "collection_partition": "unassigned",
                "source_type": "kaggle_export",
                "prompt_sha256": prompt_sha256,
                "prompt_token_count": len(prompt_ids),
                "model_pair_id": MODEL_PAIR_ID,
                "draft_model": {"model_id": DRAFT_MODEL_ID, "revision": DRAFT_MODEL_REVISION},
                "target_model": {"model_id": TARGET_MODEL_ID, "revision": TARGET_MODEL_REVISION},
                "tokenizer_source_model": {
                    "model_id": TOKENIZER_SOURCE_MODEL_ID,
                    "revision": TOKENIZER_SOURCE_MODEL_REVISION,
                },
                "seed": SEED,
                "decoding_configuration_id": DECODING_CONFIGURATION_ID,
                "decode_round": 0,
                "block_position_index": position,
                "visible_prefix_token_ids": visible_prefix_token_ids.copy(),
                "raw_draft_probability": draft_features["probability"],
                "raw_draft_entropy": draft_features["entropy"],
                "conditional_survival_confidence": draft_features["probability"],
            }
        )
        outcome_records.append(
            {
                "schema_version": OUTCOME_RECORD_SCHEMA_VERSION,
                "trace_id": trace_id,
                "case_id": case["case_id"],
                "decode_round": 0,
                "block_position_index": position,
                "candidate_token_id": candidate_token_id,
                "target_probability": target_features["probability"],
                "target_entropy": target_features["entropy"],
                "target_argmax_matches_candidate": target_argmax_matches_candidate,
                "prefix_target_argmax_match": all(target_match_history),
            }
        )
        visible_prefix_token_ids.append(candidate_token_id)
        if candidate_token_id == tokenizer.eos_token_id:
            break

    return runtime_records, outcome_records


def write_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    """Write deterministic JSONL records without raw prompt text or logits."""

    path.write_text(
        "".join(json.dumps(record, sort_keys=True) + "\n" for record in records),
        encoding="utf-8",
    )


def create_archive(source_directory: Path, archive_path: Path) -> None:
    """Create a flat, deterministic archive of the exported collection artifacts."""

    with ZipFile(archive_path, "x", compression=ZIP_DEFLATED) as archive:
        for child in sorted(source_directory.iterdir(), key=lambda item: item.name):
            archive.write(child, arcname=child.name)


def write_result(result: dict[str, Any]) -> Path:
    """Persist the terminal result whether collection passes or fails."""

    RESULT_PATH.write_text(canonical_json(result), encoding="utf-8")
    return RESULT_PATH

In [ ]:
set_seed(SEED)

result: dict[str, Any] = {
    "schema_version": RESULT_SCHEMA_VERSION,
    "collection_id": COLLECTION_ID,
    "collection_attempt_id": COLLECTION_ATTEMPT_ID,
    "status": "fails_governed_trace_collection",
    "trace_collection_performed": False,
    "trace_collection_archive_created": False,
    "source_commit_sha": None,
    "manifest_sha256": None,
    "archive_sha256": None,
    "failure": None,
}

draft_model = None
target_model = None
try:
    source_commit_sha = require_source_commit_sha()
    gpu_metadata = require_kaggle_t4()
    validate_prompt_corpus()
    require_clean_destination()

    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_SOURCE_MODEL_ID,
        revision=TOKENIZER_SOURCE_MODEL_REVISION,
        trust_remote_code=False,
        use_fast=True,
    )
    dtype = choose_dtype()
    draft_model = load_model(DRAFT_MODEL_ID, DRAFT_MODEL_REVISION, dtype)
    target_model = load_model(TARGET_MODEL_ID, TARGET_MODEL_REVISION, dtype)

    all_runtime_records: list[dict[str, Any]] = []
    all_outcome_records: list[dict[str, Any]] = []
    for case in PROMPT_CASES:
        runtime_records, outcome_records = collect_case(
            case=case,
            tokenizer=tokenizer,
            draft_model=draft_model,
            target_model=target_model,
        )
        all_runtime_records.extend(runtime_records)
        all_outcome_records.extend(outcome_records)

    if not all_runtime_records or not all_outcome_records:
        raise TraceCollectionFailure(
            "unexpected_collection_failure",
            "Trace collection produced no records.",
        )
    if len(all_runtime_records) != len(all_outcome_records):
        raise TraceCollectionFailure(
            "unexpected_collection_failure",
            "Runtime and expected-outcome record counts diverged.",
        )

    STAGING_DIRECTORY.mkdir(parents=True)
    runtime_path = STAGING_DIRECTORY / "runtime_records.jsonl"
    outcomes_path = STAGING_DIRECTORY / "expected_outcomes.jsonl"
    write_jsonl(runtime_path, all_runtime_records)
    write_jsonl(outcomes_path, all_outcome_records)

    manifest = {
        "schema_version": MANIFEST_SCHEMA_VERSION,
        "collection_id": COLLECTION_ID,
        "collection_attempt_id": COLLECTION_ATTEMPT_ID,
        "source_commit_sha": source_commit_sha,
        "preflight_attempt_id": PREFLIGHT_ATTEMPT_ID,
        "preflight_source_commit_sha": PREFLIGHT_SOURCE_COMMIT_SHA,
        "preflight_result_sha256": PREFLIGHT_RESULT_SHA256,
        "prompt_corpus_id": PROMPT_CORPUS_ID,
        "prompt_corpus_sha256": PROMPT_CORPUS_SHA256,
        "data_role": "trace_collection",
        "collection_partition": "unassigned",
        "source_type": "kaggle_export",
        "model_pair_id": MODEL_PAIR_ID,
        "draft_model": {"model_id": DRAFT_MODEL_ID, "revision": DRAFT_MODEL_REVISION},
        "target_model": {"model_id": TARGET_MODEL_ID, "revision": TARGET_MODEL_REVISION},
        "tokenizer_source_model": {
            "model_id": TOKENIZER_SOURCE_MODEL_ID,
            "revision": TOKENIZER_SOURCE_MODEL_REVISION,
        },
        "seed": SEED,
        "decoding_configuration_id": DECODING_CONFIGURATION_ID,
        "case_count": len(PROMPT_CASES),
        "runtime_record_count": len(all_runtime_records),
        "expected_outcome_record_count": len(all_outcome_records),
        "environment": environment_metadata(gpu_metadata),
        "files": [
            {
                "relative_path": "runtime_records.jsonl",
                "sha256": sha256_file(runtime_path),
                "byte_count": runtime_path.stat().st_size,
                "record_count": len(all_runtime_records),
            },
            {
                "relative_path": "expected_outcomes.jsonl",
                "sha256": sha256_file(outcomes_path),
                "byte_count": outcomes_path.stat().st_size,
                "record_count": len(all_outcome_records),
            },
        ],
    }
    manifest_path = STAGING_DIRECTORY / "manifest.json"
    manifest_path.write_text(canonical_json(manifest), encoding="utf-8")

    shutil.move(str(STAGING_DIRECTORY), str(OUTPUT_DIRECTORY))
    create_archive(OUTPUT_DIRECTORY, ARCHIVE_PATH)
    result.update(
        {
            "status": "passes_governed_trace_collection",
            "trace_collection_performed": True,
            "trace_collection_archive_created": True,
            "source_commit_sha": source_commit_sha,
            "manifest_sha256": sha256_file(OUTPUT_DIRECTORY / "manifest.json"),
            "archive_sha256": sha256_file(ARCHIVE_PATH),
        }
    )
except TraceCollectionFailure as error:
    result["failure"] = {"code": error.code, "message": safe_error_message(error)}
except Exception as error:
    result["failure"] = {
        "code": "unexpected_collection_failure",
        "message": safe_error_message(error),
    }
finally:
    del draft_model
    del target_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    persisted_path = write_result(result)

print(
    canonical_json(
        {
            "status": result["status"],
            "trace_collection_performed": result["trace_collection_performed"],
            "archive_created": result["trace_collection_archive_created"],
            "result_path": str(persisted_path),
            "result_sha256": sha256_file(persisted_path),
            "failure_code": None if result["failure"] is None else result["failure"]["code"],
        }
    )
)

if result["status"] != "passes_governed_trace_collection":
    raise RuntimeError(
        "Governed trace collection failed. The bounded result was retained; "
        "do not rerun or use partial output."
    )

print("Governed trace collection passed. Stop here and download the result JSON and archive.")

## Required retained outputs

Download both files after a pass or failure:

```text
/kaggle/working/specsafe_v5_qwen_trace_collection_result_attempt_001.json
/kaggle/working/specsafe_v5_qwen_trace_collection_v1_attempt_001.zip
```

Do not upload the archive publicly yet. Upload the result JSON here first. A successful collection authorizes only local validation and governed import design.
